# Notebook 02 — Per-Component Parameter Analysis

**Goal:** for each of the 20 vision-language models in the Phase-2 roster, count the number of parameters in each architectural component (Vision, Fusion, Language, Head) and surface the result inline so you can manually validate it against the legacy Sunita-1/2/3 numbers and against each model's reported figures.

**Inputs:** none from previous notebooks (this is the first measurement step).

**Outputs:**
* Per-model JSON records written to `../outputs/measurements/<sanitized_hf_id>.json`
* Verbose `.txt` logs in `../outputs/logs/<timestamp>_02_parameter_analysis_nb__<hf_id>.txt`
* Inline pandas DataFrames + matplotlib visualizations for spot-checking
* CSV summary at `../outputs/measurements/nb02_parameter_summary.csv`

**Compute:** CPU works; GPU helps only for faster model loading. **No memory measurements here** — those are in notebook 04.

**Disk:** the HuggingFace cache will grow to ~250–500 GB by the end of the full 20-model run. Colab's free tier (~235 GB) **cannot fit all 20 models without intermediate cleanup**. Set `CLEANUP_HF_CACHE_BETWEEN_MODELS = True` in §1 below if running on Colab. On Runpod with 1 TB+ disk, set to `False` to keep the cache for later notebooks.

**How to read this notebook:** run top-to-bottom. Each section has a markdown header followed by code cells. Look at the DataFrame after each cell to verify the numbers make sense before moving on.

**Revisions:** v2 (2026-05-21) adds disk-space monitoring, inline natural-language interpretation, four visualization cells, family-level summaries, and a dedicated investigation cell for the LLaVA-1.5 ~152M discrepancy with Sunita-1.

---
## 1. Setup

Run **one** of the setup cells below depending on where you're running the notebook. Set the disk-cleanup flag based on your environment.

In [ ]:
# ===== Environment-specific flags =====
# Set to True on Colab (free tier ~235 GB disk) to delete each model's HF
# cache after measuring its params. Set to False on Runpod (1TB+ disk) so
# subsequent notebooks (03 FLOPs, 04 memory) can reuse the cache.
CLEANUP_HF_CACHE_BETWEEN_MODELS = False

# Warn when free disk drops below this many GB. Aborts the loop if it goes lower.
DISK_FREE_GB_WARN = 30
DISK_FREE_GB_ABORT = 10

# Set this to MODELS[:3] for a smoke test; MODELS for the full 20-model run.
# (this is just the default; the main-loop cell lets you override.)
DEFAULT_TARGETS_SLICE = slice(None)  # i.e. MODELS[:] = all 20

In [ ]:
# ===== Setup option A: Colab =====
# Uncomment the four lines below if running in Google Colab.
#
# from google.colab import drive, userdata
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/Sunita\ PhD\ Work/Thread_B_Architectural_Asymmetry/code_active
# !pip install -q -r requirements.txt
# import os; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# ===== Setup option B: Runpod / institute box =====
# (Assumes you've cd'd into code_active/ and set HF_TOKEN before launching jupyter)
# !pip install -q -r ../requirements.txt

# ===== Setup option C: local Windows / OneDrive (your machine) =====
import os
# os.environ['HF_TOKEN'] = '...'  # if not already in env

In [ ]:
import sys, shutil
from pathlib import Path

NB_DIR = Path.cwd()
CODE_ACTIVE = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(CODE_ACTIVE))

MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
LOG_DIR = CODE_ACTIVE / 'outputs' / 'logs'
FIG_DIR = CODE_ACTIVE / 'outputs' / 'figures'
for d in (MEASUREMENT_DIR, LOG_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

HF_CACHE = Path(os.environ.get('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))).expanduser() / 'hub'

def free_disk_gb(p='/'):
    s = shutil.disk_usage(p)
    return s.free / 1e9, s.total / 1e9

free_gb, total_gb = free_disk_gb()
print(f'Working dir:        {NB_DIR}')
print(f'code_active dir:    {CODE_ACTIVE}')
print(f'vlm_audit:          {CODE_ACTIVE / "vlm_audit"}')
print(f'HF cache dir:       {HF_CACHE}')
print(f'Disk:               {free_gb:.1f} GB free / {total_gb:.1f} GB total ({100*free_gb/total_gb:.1f}% free)')
if free_gb < 50:
    print(f'⚠ Less than 50 GB free — set CLEANUP_HF_CACHE_BETWEEN_MODELS=True or move HF_HOME to a bigger volume.')

In [ ]:
import pandas as pd
import numpy as np
import torch
import time
import traceback
import matplotlib.pyplot as plt

from vlm_audit import (
    MODELS, get_model_list, count_params,
    extract_components, compute_trainable_from_policy,
    run_full_audit, InputConfig,
)
from vlm_audit.runner import _load_model, _resolve_hf_class
from vlm_audit.components import freezing_policy_hint, trainable_components_for_family
from vlm_audit.logging_utils import sanitize_for_filename

print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')
    print(f'VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Models in roster: {len(MODELS)}')
print(f'Cleanup HF cache between models: {CLEANUP_HF_CACHE_BETWEEN_MODELS}')

---
## 2. Inspect the model roster

Before measuring anything, show the 20 models we're about to audit. Sanity-check that the roster is what you expect, and that total fp16 footprint sums to something tractable for your disk.

In [ ]:
roster_df = pd.DataFrame([
    {
        'name':           m.name,
        'family':         m.family,
        'year':           m.release_year,
        'fp16_GB':        m.fits_in_gb_fp16,
        'native_HW':      f'{m.native_image_hw[0]}x{m.native_image_hw[1]}',
        'recent_gen':     m.recent_gen,
        'is_moe':         m.is_moe,
        'trust_remote':   m.requires_trust_remote_code,
        'hf_id':          m.hf_id,
    }
    for m in MODELS
])
total_fp16_GB = roster_df['fp16_GB'].sum()
print(f'Total models: {len(roster_df)}')
print(f'Families:     {sorted(roster_df.family.unique())} ({len(roster_df.family.unique())} families)')
print(f'Recent-gen models (2024+): {int(roster_df.recent_gen.sum())}')
print(f'Total fp16 footprint sum (lower bound, no activations): {total_fp16_GB:.1f} GB')
print(f'Cumulative if all models cached on disk: roughly 2x = {2*total_fp16_GB:.0f} GB.')
roster_df

---
## 3. Smoke test on CLIP-ViT-L/14 (validation against legacy Sunita-1)

Before iterating over all 20 models, validate the pipeline on **one** known-good model where we have a ground-truth comparison.

From the legacy `Sunita-1.ipynb` cell 12, CLIP-ViT-L/14 had:
* `vision_model`:  303,179,776 params
* `text_model`:    123,060,480 params
* projection:        1,376,256 params
* **Total:**       427,616,512 params

If our pipeline matches these to the last digit, we know `extract_components` and `count_params` are correct for the CLIP family.

In [ ]:
clip_spec = next(m for m in MODELS if m.name == 'CLIP-ViT-L/14')
print(f'Auditing: {clip_spec.name}')
print(f'  HF id:     {clip_spec.hf_id}')
print(f'  Family:    {clip_spec.family}')
print(f'  Native HW: {clip_spec.native_image_hw}')
print(f'  Freezing recipe: {freezing_policy_hint(clip_spec.family)}')
print(f'  Trainable-component table per recipe: {trainable_components_for_family(clip_spec.family)}')

In [ ]:
t0 = time.time()
rec = run_full_audit(
    clip_spec,
    metrics={'params'},
    measurement_dir=str(MEASUREMENT_DIR),
    log_dir=str(LOG_DIR),
    notebook_label='02_parameter_analysis_nb',
)
print(f'Done in {time.time() - t0:.1f}s')
print(f'Log file: {sorted(LOG_DIR.glob("*02_parameter_analysis_nb__openai*"))[-1].name}')

In [ ]:
clip_df = pd.DataFrame([
    {
        'component':         c.name,
        'params_total':      c.params_total,
        'params_total_M':    round(c.params_total / 1e6, 2),
        'params_trainable':  c.params_trainable,
        'trainable_frac':    round(c.params_trainable / max(c.params_total, 1), 3),
        'notes':             c.notes,
    }
    for c in rec.components.values()
])
clip_df

In [ ]:
EXPECTED = {
    'vision':   303_179_776,
    'language': 123_060_480,
    'fusion':     1_376_256,
}
ok = True
for cname, expected in EXPECTED.items():
    got = rec.components[cname].params_total
    delta = got - expected
    status = '✓' if delta == 0 else f'✗ (off by {delta:+,})'
    print(f'  {cname:10s}: expected {expected:>12,}  got {got:>12,}  {status}')
    if delta != 0:
        ok = False
assert ok, 'CLIP-L params do not match legacy Sunita-1 — extractor may be broken!'
print('\n✓ CLIP-L params match Sunita-1 cell 12 exactly. Pipeline is correct for CLIP family.')
print('  Interpretation: CLIP-L is vision-heavy — 70.9% vision, 28.8% language, 0.3% fusion.')
print('                  V/L ratio = 2.46×. This is the corrected ratio (not the 10× from older docs — see memory.md M-001).')

---
## 4. Tricky model verification — IDEFICS-9B (Phase-2 F-3 bug fix)

IDEFICS-9B was the Phase-1 bug case: its `language` component used to read 0 params because the extractor couldn't find the LLaMA backbone. Phase-2 components.py fixes this by explicitly walking into `model.model` and assembling a ModuleList of the LLaMA layers + embed_tokens + norm.

Validate the fix before scaling to all 20:
* **Phase-1 (broken):** language = 0 → IDEFICS labeled 83% vision-dominant (wrong)
* **Phase-2 (correct):** language ≈ 6.6B → IDEFICS correctly labeled language-dominant

In [ ]:
idefics_spec = next(m for m in MODELS if m.name == 'IDEFICS-9B')
rec_idefics = run_full_audit(
    idefics_spec, metrics={'params'},
    measurement_dir=str(MEASUREMENT_DIR),
    log_dir=str(LOG_DIR),
    notebook_label='02_parameter_analysis_nb',
)
idefics_df = pd.DataFrame([
    {'component': c.name, 'params_M': round(c.params_total/1e6, 2),
     'trainable_M': round(c.params_trainable/1e6, 2),
     'trainable_pct': round(100 * c.params_trainable / max(c.params_total, 1), 1),
     'notes': c.notes}
    for c in rec_idefics.components.values()
])
idefics_df

In [ ]:
# Verify F-3 + F-5 fixes simultaneously
lang_params = rec_idefics.components['language'].params_total
lang_trainable = rec_idefics.components['language'].params_trainable
vision_trainable = rec_idefics.components['vision'].params_trainable
fusion_trainable = rec_idefics.components['fusion'].params_trainable

print('=== Phase-2 F-3 fix verification (IDEFICS language extractor) ===')
print(f'  language params: {lang_params:,} ({lang_params/1e9:.2f}B)')
assert lang_params > 5_000_000_000, f'F-3 fix not working — language should be ~6.6B, got {lang_params:,}'
print('  ✓ Phase-2 F-3 fix WORKS — IDEFICS language is now correctly ~6.6B (was 0 in Phase 1)')

print('\n=== Phase-2 F-5 fix verification (trainable from policy, not requires_grad) ===')
print(f'  vision_trainable:   {vision_trainable:,}  (recipe: frozen → expect 0)')
print(f'  language_trainable: {lang_trainable:,}  (recipe: frozen → expect 0)')
print(f'  fusion_trainable:   {fusion_trainable:,}  (recipe: gated cross-attn trainable → expect ~126M)')
assert vision_trainable == 0, f'F-5 fix not working — vision should show 0 trainable (frozen), got {vision_trainable:,}'
assert lang_trainable == 0, f'F-5 fix not working — language should show 0 trainable (frozen), got {lang_trainable:,}'
assert fusion_trainable > 100_000_000, f'F-5 fix not working — fusion should be trainable ~126M, got {fusion_trainable:,}'
print('  ✓ Phase-2 F-5 fix WORKS — IDEFICS shows frozen vision + frozen language + trainable cross-attention')

print('\nInterpretation: IDEFICS-9B trains only 257M parameters (fusion + head) out of 7.5B total — 3.4% trainable.')
print('This is the canonical Flamingo-style design: massive frozen vision + massive frozen LLM, learn only the bridge.')

---
## 5. Run parameter counts on all 20 models

Iterate over `MODELS`, wrap each in try/except so one failure doesn't kill the whole run. Failures collected into `failures_df`.

Per-model output:
* `[i/N] <name>` header
* progress: load time and total params on success
* **inline interpretation**: short natural-language summary of the V/F/L/H allocation
* **disk monitoring**: free space before and after each model; cleanup if `CLEANUP_HF_CACHE_BETWEEN_MODELS = True`
* `failures_df` if anything blew up

**Disk planning:** model 9 (Idefics2-8B) on Colab took 12+ minutes due to bandwidth bottleneck — be patient. Model 14 (InternVL3-8B) is where Colab free-tier disk usually fills up. Set `CLEANUP_HF_CACHE_BETWEEN_MODELS = True` to keep going.

**Wall-clock estimate:**
* Runpod L40 + 1TB disk + good bandwidth: **~20-40 minutes** for all 20 models
* Colab L4 + 235GB disk (with cleanup): **~60-90 minutes**

In [ ]:
# Helpers used inside the main loop below.
import gc

def hf_cache_size_gb(hf_id: str) -> float:
    cache_subdir = HF_CACHE / f'models--{hf_id.replace("/", "--")}'
    if not cache_subdir.exists():
        return 0.0
    total = 0
    for p in cache_subdir.rglob('*'):
        if p.is_file() and not p.is_symlink():
            try:
                total += p.stat().st_size
            except Exception:
                pass
    return total / 1e9

def cleanup_hf_cache_for(hf_id: str, log=True) -> float:
    cache_subdir = HF_CACHE / f'models--{hf_id.replace("/", "--")}'
    if not cache_subdir.exists():
        if log:
            print(f'      [cleanup] no cache for {hf_id} — skipping')
        return 0.0
    sz = hf_cache_size_gb(hf_id)
    shutil.rmtree(cache_subdir, ignore_errors=True)
    if log:
        print(f'      [cleanup] removed {sz:.2f} GB of cache for {hf_id}')
    return sz

def interpret_allocation(rec) -> str:
    v = (rec.components.get('vision').params_total if rec.components.get('vision') else 0) or 0
    f = (rec.components.get('fusion').params_total if rec.components.get('fusion') else 0) or 0
    l = (rec.components.get('language').params_total if rec.components.get('language') else 0) or 0
    total = v + f + l
    if total == 0:
        return '      [interpret] no V/F/L decomposition (native model — see notebook 10)'
    v_pct = 100 * v / total
    f_pct = 100 * f / total
    l_pct = 100 * l / total
    if v_pct > 50:
        dom = 'vision-dominant'
    elif l_pct > 70:
        dom = 'language-dominant'
    elif 0.4 < v / max(l, 1) < 2.5:
        dom = 'roughly balanced'
    else:
        dom = 'mixed'
    trainable_total = sum((c.params_trainable or 0) for c in rec.components.values())
    total_w_head = sum((c.params_total or 0) for c in rec.components.values())
    train_pct = 100 * trainable_total / max(total_w_head, 1)
    return (f'      [interpret] {v_pct:.1f}% vision · {f_pct:.1f}% fusion · {l_pct:.1f}% language '
            f'→ {dom}; {train_pct:.1f}% trainable under canonical recipe')

In [ ]:
# Choose what to run:
#   MODELS         — all 20 (full run)
#   MODELS[:3]     — smoke test
#   MODELS[10:]    — only Phase-2 new additions
#   MODELS[13:]    — only models 14-20 (e.g. if Colab stopped at 13)
TARGETS = MODELS

results = []
failures = []
cleanup_total_gb = 0.0

for i, spec in enumerate(TARGETS, 1):
    free_gb_before, _ = free_disk_gb()
    print(f'\n[{i}/{len(TARGETS)}] {spec.name:30s} ({spec.hf_id})  disk_free={free_gb_before:.1f} GB')

    if free_gb_before < DISK_FREE_GB_ABORT:
        print(f'    ⛔ disk free < {DISK_FREE_GB_ABORT} GB — ABORTING. set CLEANUP_HF_CACHE_BETWEEN_MODELS=True or move HF_HOME.')
        failures.append({'name': spec.name, 'error': f'aborted: disk_free={free_gb_before:.1f} GB < {DISK_FREE_GB_ABORT}'})
        break
    elif free_gb_before < DISK_FREE_GB_WARN:
        print(f'    ⚠ disk free < {DISK_FREE_GB_WARN} GB — proceeding, but watch')

    t0 = time.time()
    try:
        rec = run_full_audit(
            spec, metrics={'params'},
            measurement_dir=str(MEASUREMENT_DIR),
            log_dir=str(LOG_DIR),
            notebook_label='02_parameter_analysis_nb',
        )
        row = {'name': spec.name, 'family': spec.family, 'year': spec.release_year}
        for cname, c in rec.components.items():
            row[f'{cname}_total'] = c.params_total or 0
            row[f'{cname}_trainable'] = c.params_trainable or 0
        results.append(row)
        total_M = sum((c.params_total or 0) for c in rec.components.values()) / 1e6
        print(f'    OK  total={total_M:.2f}M params  ({time.time()-t0:.1f}s)')
        print(interpret_allocation(rec))
        if CLEANUP_HF_CACHE_BETWEEN_MODELS:
            cleanup_total_gb += cleanup_hf_cache_for(spec.hf_id, log=True)
    except Exception as e:
        tb = traceback.format_exc()
        failures.append({
            'name': spec.name, 'family': spec.family,
            'error': f'{type(e).__name__}: {e}',
            'traceback_excerpt': '\n'.join(tb.splitlines()[-5:]),
        })
        print(f'    FAILED  {type(e).__name__}: {e}')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'\n=== Finished: {len(results)} succeeded, {len(failures)} failed ===')
if CLEANUP_HF_CACHE_BETWEEN_MODELS:
    print(f'  Cleaned up {cleanup_total_gb:.1f} GB of HF cache during run')
free_gb_after, _ = free_disk_gb()
print(f'  Disk free now: {free_gb_after:.1f} GB')

---
## 6. Master parameter table

Display the full V/F/L/H breakdown for all 20 models in one DataFrame, sorted by family then year. This is the table that becomes Table 1 in the paper.

Native-fusion models (Chameleon) have additional component columns (`fused_transformer`, `embeddings`); decoupled-encoder models (Janus-Pro) have `vision_und`, `vision_gen`. These are intentionally separate from the V/F/L/H aggregate because they don't fit the same schema — they're handled in notebook 10.

In [ ]:
if not results:
    print('No successful results — see failures below.')
    params_df = pd.DataFrame()
else:
    params_df = pd.DataFrame(results)
    for col in [c for c in params_df.columns if c.endswith('_total') or c.endswith('_trainable')]:
        params_df[col + '_M'] = (params_df[col] / 1e6).round(2)
    keep_cols = ['name', 'family', 'year']
    for c in ['vision', 'fusion', 'language', 'head']:
        if f'{c}_total_M' in params_df.columns:
            keep_cols.append(f'{c}_total_M')
    display_df = params_df[keep_cols].sort_values(['family', 'year']).reset_index(drop=True)
display_df if results else None

In [ ]:
# Annotated view with derived columns
if results:
    params_df['total_M'] = (
        params_df.get('vision_total', 0).fillna(0) +
        params_df.get('fusion_total', 0).fillna(0) +
        params_df.get('language_total', 0).fillna(0) +
        params_df.get('head_total', 0).fillna(0)
    ) / 1e6
    params_df['vision_pct']   = (100 * params_df.get('vision_total', 0).fillna(0)   / (params_df['total_M'] * 1e6).clip(lower=1)).round(1)
    params_df['fusion_pct']   = (100 * params_df.get('fusion_total', 0).fillna(0)   / (params_df['total_M'] * 1e6).clip(lower=1)).round(1)
    params_df['language_pct'] = (100 * params_df.get('language_total', 0).fillna(0) / (params_df['total_M'] * 1e6).clip(lower=1)).round(1)
    params_df['v_to_l_ratio'] = (
        params_df.get('vision_total', 0).fillna(0) /
        params_df.get('language_total', 1).replace(0, float('nan'))
    ).round(3)
    # Trainable totals + fraction
    params_df['trainable_total_M'] = sum(
        params_df.get(f'{c}_trainable', 0).fillna(0) for c in ['vision', 'fusion', 'language', 'head']
    ) / 1e6
    params_df['trainable_pct'] = (100 * params_df['trainable_total_M'] / params_df['total_M'].clip(lower=1e-6)).round(1)
summary_cols = ['name', 'family', 'year', 'total_M', 'vision_pct', 'fusion_pct', 'language_pct', 'v_to_l_ratio', 'trainable_total_M', 'trainable_pct']
params_df[summary_cols].sort_values(['family', 'year']).reset_index(drop=True) if results else None

**Reading the table above:**
* CLIP/BLIP/SigLIP era models (2021-2023 contrastive + bridge) → **vision-heavy** (vision_pct > 50%).
* LLaVA / Qwen-VL / Idefics2 era models (2023-2025 adapter-LLM) → **language-heavy** (language_pct > 80%).
* SigLIP-L (V/L=0.94) vs SigLIP2-So400m/14: does the 2025 update stay balanced?
* IDEFICS-9B: post Phase-2 F-3 fix, language_pct ≈ 88% (NOT 0% like Phase 1).
* `trainable_pct`: ranges from ~3-4% (BLIP-2, IDEFICS, Idefics2 — frozen vision + frozen LLM) to 100% (CLIP, SigLIP, ViLT — end-to-end).

If any row doesn't look like this, flag the model and look at its `outputs/logs/*02_parameter_analysis_nb__<that_model>.txt` for the traceback.

---
## 7. Visualizations — params at a glance

Four headline charts:
1. **Total params bar chart** (log y) — see the 70× range from ViLT (117M) to Idefics2 (8.4B).
2. **Stacked V/F/L/H bars** — visualize allocation pattern per model.
3. **V/L ratio by family** — the family-dependent asymmetry headline.
4. **Trainable vs total** — visualize the freezing argument (F5).

In [ ]:
if results:
    plot_df = params_df.copy().sort_values('total_M').reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(plot_df['name'], plot_df['total_M'], color='steelblue', edgecolor='black')
    ax.set_xscale('log')
    ax.set_xlabel('Total parameters (M, log scale)')
    ax.set_title('Total params per model — Phase-2 n=20 roster')
    for bar, val in zip(bars, plot_df['total_M']):
        ax.annotate(f'{val:,.0f}M', xy=(bar.get_width(), bar.get_y() + bar.get_height()/2),
                    xytext=(5, 0), textcoords='offset points', va='center', fontsize=7)
    ax.grid(True, which='both', axis='x', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'nb02_total_params_bar.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {FIG_DIR / "nb02_total_params_bar.png"}')

In [ ]:
if results:
    # Stacked bar V/F/L/H per model, sorted by family then year
    sb_df = params_df.copy().sort_values(['family', 'year', 'name']).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(14, 8))
    x = np.arange(len(sb_df))
    width = 0.7
    colors = {'vision': '#2ECC71', 'fusion': '#E67E22', 'language': '#3498DB', 'head': '#95A5A6'}
    bottoms = np.zeros(len(sb_df))
    for cname in ['vision', 'fusion', 'language', 'head']:
        col = f'{cname}_total'
        if col not in sb_df.columns:
            continue
        vals = (sb_df[col].fillna(0) / 1e6).values
        ax.bar(x, vals, width=width, bottom=bottoms, color=colors[cname], edgecolor='black', linewidth=0.5, label=cname)
        bottoms += vals
    ax.set_xticks(x)
    ax.set_xticklabels(sb_df['name'], rotation=60, ha='right', fontsize=9)
    ax.set_ylabel('Parameters (M)')
    ax.set_yscale('log')
    ax.set_title('Per-component parameter breakdown — V/F/L/H stacked bars, log y')
    ax.legend(loc='upper left', framealpha=0.9)
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'nb02_stacked_vflh.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {FIG_DIR / "nb02_stacked_vflh.png"}')

In [ ]:
if results:
    # V/L ratio by family (log y) + vision % overlay
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    # Left panel: V/L ratio (log y)
    rdf = params_df.dropna(subset=['v_to_l_ratio']).copy()
    rdf = rdf[rdf['v_to_l_ratio'] > 0].sort_values('v_to_l_ratio')
    bars = axes[0].barh(rdf['name'], rdf['v_to_l_ratio'], color='coral', edgecolor='black')
    axes[0].axvline(1.0, color='black', linestyle='--', alpha=0.5, label='V = L (balanced)')
    axes[0].set_xscale('log')
    axes[0].set_xlabel('Vision / Language parameter ratio (log)')
    axes[0].set_title('V/L ratio per model')
    axes[0].legend()
    axes[0].grid(True, which='both', axis='x', linestyle='--', alpha=0.3)
    for bar, val in zip(bars, rdf['v_to_l_ratio']):
        axes[0].annotate(f'{val:.3f}', xy=(bar.get_width(), bar.get_y() + bar.get_height()/2),
                         xytext=(3, 0), textcoords='offset points', va='center', fontsize=7)

    # Right panel: vision% per family (box plot if family has >1 model, else dot)
    fam_summary = params_df.groupby('family').agg(
        n=('name', 'count'),
        avg_vision_pct=('vision_pct', 'mean'),
        avg_language_pct=('language_pct', 'mean'),
    ).sort_values('avg_vision_pct', ascending=False)
    axes[1].barh(fam_summary.index, fam_summary['avg_vision_pct'], color='#2ECC71', edgecolor='black', label='vision %')
    axes[1].barh(fam_summary.index, fam_summary['avg_language_pct'], left=fam_summary['avg_vision_pct'],
                 color='#3498DB', edgecolor='black', label='language %')
    axes[1].set_xlim(0, 105)
    axes[1].set_xlabel('Mean parameter share within family (%)')
    axes[1].set_title('Average V vs L share by architectural family')
    axes[1].legend(loc='lower right')
    for i, (idx, row) in enumerate(fam_summary.iterrows()):
        axes[1].annotate(f'n={int(row["n"])}', xy=(101, i), va='center', fontsize=7)
    axes[1].grid(True, axis='x', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'nb02_vlratio_and_family.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {FIG_DIR / "nb02_vlratio_and_family.png"}')

In [ ]:
if results:
    # Trainable vs total — the freezing visualization (F5)
    sb_df = params_df.copy().sort_values(['family', 'year']).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(sb_df))
    width = 0.35
    ax.bar(x - width/2, sb_df['total_M'], width=width, color='lightgray', edgecolor='black', label='Total params')
    ax.bar(x + width/2, sb_df['trainable_total_M'], width=width, color='#E74C3C', edgecolor='black', label='Trainable (canonical recipe)')
    ax.set_yscale('log')
    ax.set_xticks(x)
    ax.set_xticklabels(sb_df['name'], rotation=60, ha='right', fontsize=9)
    ax.set_ylabel('Parameters (M, log scale)')
    ax.set_title('Total vs trainable parameters — the Phase-2 F-5 fix in action\n'
                 'tall gray bars + short red bars = lots of frozen capacity (BLIP-2, IDEFICS, Idefics2, LLaVA family)')
    ax.legend(loc='upper left')
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)
    # Annotate trainable %
    for i, (idx, row) in enumerate(sb_df.iterrows()):
        ax.annotate(f'{row["trainable_pct"]:.0f}%', xy=(i, row['trainable_total_M']),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=7, color='#E74C3C')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'nb02_trainable_vs_total.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {FIG_DIR / "nb02_trainable_vs_total.png"}')

---
## 8. Per-family summary statistics

Aggregate by family. This is one of the headline tables for the paper.

In [ ]:
if results:
    family_summary = params_df.groupby('family').agg(
        n_models       = ('name', 'count'),
        avg_total_M    = ('total_M', 'mean'),
        avg_vision_pct = ('vision_pct', 'mean'),
        avg_fusion_pct = ('fusion_pct', 'mean'),
        avg_language_pct = ('language_pct', 'mean'),
        avg_v_to_l_ratio = ('v_to_l_ratio', 'mean'),
        avg_trainable_pct = ('trainable_pct', 'mean'),
        models = ('name', lambda s: ', '.join(s)),
    ).round(1).sort_values('avg_vision_pct', ascending=False)
family_summary if results else None

In [ ]:
# Cross-family interpretation
if results:
    print('=== Family-level allocation patterns ===\n')
    print('Vision-dominant families (mean vision_pct > 50%):')
    vd = family_summary[family_summary['avg_vision_pct'] > 50]
    for fam, row in vd.iterrows():
        print(f'  {fam:35s} mean vision = {row["avg_vision_pct"]:.1f}%  (n={int(row["n_models"])})')
    print('\nLanguage-dominant families (mean language_pct > 70%):')
    ld = family_summary[family_summary['avg_language_pct'] > 70]
    for fam, row in ld.iterrows():
        print(f'  {fam:35s} mean language = {row["avg_language_pct"]:.1f}%  (n={int(row["n_models"])})')
    print('\nBalanced families (neither dominates):')
    bal = family_summary[(family_summary['avg_vision_pct'] <= 50) & (family_summary['avg_language_pct'] <= 70)]
    for fam, row in bal.iterrows():
        print(f'  {fam:35s} V={row["avg_vision_pct"]:.1f}%  L={row["avg_language_pct"]:.1f}%  (n={int(row["n_models"])})')

---
## 9. Failures (debugging)

If any model failed, the row appears here. Look at the corresponding log file in `../outputs/logs/` for the full traceback.

In [ ]:
if failures:
    failures_df = pd.DataFrame(failures)
    print(f'{len(failures)} failures:')
    failures_df
else:
    print('No failures — all models succeeded.')

---
## 10. Sanity checks against legacy Sunita-1/2/3 numbers

Cross-reference against the legacy notebooks. **Note:** the 2026-05-21 Colab run found a ~152M discrepancy on LLaVA-1.5 (Sunita-1: 7,063.43M, Phase-2: 6,911.11M). The next cell investigates.

In [ ]:
LEGACY_CHECKS = [
    # (model name, component, expected param count, source, delta_pct_tolerance)
    ('CLIP-ViT-L/14',     'vision',     303_179_776,   'Sunita-1 cell 12', 0.0),
    ('CLIP-ViT-L/14',     'language',   123_060_480,   'Sunita-1 cell 12', 0.0),
    ('CLIP-ViT-L/14',     'fusion',       1_376_256,   'Sunita-1 cell 12', 0.0),
    ('LLaVA-1.5-7B',      'vision',     303_507_456,   'Sunita-1 cell 9',  0.0),
    ('LLaVA-1.5-7B',      'fusion',      20_979_712,   'Sunita-1 cell 9 (projector)',  0.0),
    ('LLaVA-1.5-7B',      'language',  6_738_939_904,  'Sunita-1 cell 9',  3.0),  # tolerance to handle weight-tying
]

if results:
    rows = []
    for name, comp, expected, source, tol_pct in LEGACY_CHECKS:
        match = params_df[params_df['name'] == name]
        if match.empty:
            rows.append({'model': name, 'component': comp, 'expected': expected, 'got': None,
                          'delta': None, 'pct_off': None, 'status': 'NOT RUN', 'source': source})
            continue
        col = f'{comp}_total'
        got = int(match.iloc[0].get(col, 0) or 0)
        delta = got - expected
        pct_off = 100 * abs(delta) / max(expected, 1)
        if delta == 0:
            status = '✓ exact'
        elif pct_off <= tol_pct:
            status = f'✓ within {tol_pct}% tolerance'
        else:
            status = f'⚠ OFF by {delta:+,} ({pct_off:.2f}%)'
        rows.append({'model': name, 'component': comp, 'expected': expected, 'got': got,
                      'delta': delta, 'pct_off': round(pct_off, 2), 'status': status, 'source': source})
    pd.DataFrame(rows)

### 10b. Investigation — LLaVA-1.5 ~152M discrepancy with Sunita-1

From the 2026-05-21 Colab run:
* Sunita-1 cell 9 (legacy):  LLaVA-1.5 total = **7,063,427,072** (303.5M vision + 21.0M projector + 6,738.9M LLM, head=0)
* Phase-2 (2026-05-21):       LLaVA-1.5 total = **6,911,113,728** — 152M short

Most likely cause: HuggingFace `LlavaForConditionalGeneration` may now have `tie_word_embeddings=True` so the `lm_head` weights are shared with the input embedding table. The previous Sunita-1 count may have double-counted them. The next cell loads LLaVA-1.5 directly and checks.

In [ ]:
# Optional investigation — uncomment to load LLaVA-1.5 and check tied-weights status.
# Requires ~16 GB of GPU memory; skip if Pixtral / DeepSeek already loaded.
#
# from transformers import LlavaForConditionalGeneration
# llava_spec = next(m for m in MODELS if m.name == 'LLaVA-1.5-7B')
# model = LlavaForConditionalGeneration.from_pretrained(
#     llava_spec.hf_id, dtype=torch.float16, low_cpu_mem_usage=True
# )
# print(f'tie_word_embeddings: {model.config.tie_word_embeddings if hasattr(model.config, "tie_word_embeddings") else "N/A"}')
# print(f'Total params (sum p.numel()): {sum(p.numel() for p in model.parameters()):,}')
# print(f'Total unique params (deduped by tied weight tensor id): '
#       f'{sum(p.numel() for _, p in {id(p): p for p in model.parameters()}.items()):,}')
# print('\nlanguage_model submodule:')
# print(model.language_model)
# print(f'\nlm_head.weight is embed_tokens.weight? '
#       f'{model.language_model.lm_head.weight.data_ptr() == model.language_model.model.embed_tokens.weight.data_ptr()}')
# del model; gc.collect(); torch.cuda.empty_cache()
print('(See cell source. Run only if you want to dig into the 152M discrepancy.)')

---
## 11. Save the master parameter table to disk

The per-model JSONs are already saved (incrementally merged by `run_full_audit`). We additionally save the aggregated DataFrame as CSV + the family summary.

In [ ]:
if not results:
    print('Nothing to save — no successful results.')
else:
    csv_master = MEASUREMENT_DIR / 'nb02_parameter_summary.csv'
    csv_family = MEASUREMENT_DIR / 'nb02_family_summary.csv'
    params_df.to_csv(csv_master, index=False)
    family_summary.to_csv(csv_family)
    print(f'Saved: {csv_master}')
    print(f'Saved: {csv_family}')
    print(f'Per-model JSONs: {MEASUREMENT_DIR}/<hf_id>.json')
    print(f'Verbose .txt logs: {LOG_DIR}/')
    print(f'Figures: {FIG_DIR}/nb02_*.png')
    print()
    print(f'Models with per-model JSONs ({len(list(MEASUREMENT_DIR.glob("*.json")))} files):')
    for j in sorted(MEASUREMENT_DIR.glob('*.json')):
        print(f'  {j.name}')

---
## Next steps

* If everything above looks right → continue to **`03_flops_analysis.ipynb`** for FLOPs measurements.
* If a model failed → open `outputs/logs/<timestamp>_02_parameter_analysis_nb__<that_model>.txt`. Common causes:
  * Missing `HF_TOKEN` for gated models (LLaMA / IDEFICS / Pixtral / Llama-3.2-Vision)
  * `trust_remote_code` not allowed for InternVL3 / Molmo / DeepSeek-VL2 / Janus-Pro
  * OOM on Pixtral-12B → pass `quantization='int8'` in the run loop
  * Disk full → set `CLEANUP_HF_CACHE_BETWEEN_MODELS = True` at the top, restart kernel, re-run
* If a number looks wrong → use `01_inspect_a_single_model.ipynb` to load the model interactively and walk the module tree.

*End of notebook 02 (v2 — 2026-05-21).*